# F02-P4 Pathway

Reads the pre-computed pathway raster (canonical_v3) and reports the area and share of each NBS
pathway across the AOI. This notebook summarises a finished raster; it does not assign pathways.

**canonical_v3 band layout**, not v2: band 1 pathway, band 2 ecosystem, band 3 cat_code. The v2
raster (band 2 = secondary pathway, band 3 = ecosystem) is incompatible and must not be used.

Writes `outputs/<aoi_id>__F02-P4-pathway.json`.

## Setup

In [6]:
%load_ext autoreload
%autoreload 2

from dataclasses import dataclass

import geopandas as gpd
import numpy as np

from config import *
from common import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
AOI_PATH = r"E:\NBSTOOLV3\AOI1.shp"
aoi_id = "aoi1"

aoi = prepare_aoi(gpd.read_file(AOI_PATH))
print(f"AOI {aoi_id}: {fmt_ha(aoi.area_ha)}")

results: dict[str, ComponentResult] = {}

AOI aoi1: 67,439 ha


---
## 4.1 Pathway Distribution

Area (ha) and share of the AOI for each pathway.

**Data.** `pathway.tif`, three bands.

| Band | Name | Codes |
|---|---|---|
| 1 | `pathway` | 0 no data, 1 Protect, 2 Manage, 3 Restore, 4 Ineligible |
| 2 | `ecosystem` | 0 none, 1 dryland forest, 2 mangrove, 3 peatland, 4 savanna |
| 3 | `cat_code` | 1 to 17 category index |

Code 4 Ineligible is not a pathway (established plantation, forest-lost savanna, stable savanna,
settlement); it is listed but excluded from the eligible headline. Denominator is the total AOI
area, with an Unclassified row for code 0 and nodata, so the table sums to 100. Bands 2 and 3 are
not tabulated; their codes-present pass through `values` for the F02-P5 activity join on
`(cat_code, ecosystem)`.

**Example render.**

> Of the 1,240 ha project area, 1,190 ha (96%) qualifies for a Nature-Based Solutions pathway.
> Manage covers the largest share at 620 ha (50%), followed by Restore at 370 ha (30%) and
> Protect at 200 ha (16%). A further 50 ha (4%) is ineligible.

In [8]:
@dataclass(frozen=True)
class PathwayShare:
    """One row of the pathway breakdown."""

    code: int
    label: str
    area_ha: float
    pct: float          # share of the TOTAL AOI area, not of the classified area
    is_pathway: bool    # False for Ineligible and Unclassified


UNCLASSIFIED_LABEL = "Unclassified"


def _codes_present(aoi: AOI, band: int, labels: dict[int, str]) -> dict:
    """Read one pass-through band and return which codes occur, with labels and areas.

    Used for the ecosystem and cat_code bands. They are not tabulated into the headline, but the
    activity generator needs to know which categories the AOI contains, so the set of present
    codes and their areas travel in `values`.
    """
    raster = load_raster_clipped(PATHWAY_RASTER, aoi, resampling="nearest", band=band)
    codes, counts = np.unique(raster.values.compressed(), return_counts=True)
    area_by_code = {
        int(c): int(n) * raster.pixel_area_ha
        for c, n in zip(codes.tolist(), counts.tolist())
        if int(c) != 0  # 0 is mask / none, not a category
    }
    return {
        "codes": sorted(area_by_code),
        "labels": [labels.get(c, f"Unknown code {c}") for c in sorted(area_by_code)],
        "area_ha": {labels.get(c, f"Unknown code {c}"): area_by_code[c]
                    for c in sorted(area_by_code)},
    }


def analyze_pathway_distribution(aoi: AOI) -> ComponentResult:
    """Component 4.1. Area and share of the AOI per primary pathway, canonical_v3."""
    primary = load_raster_clipped(
        PATHWAY_RASTER, aoi, resampling="nearest", band=PATHWAY_BAND
    )

    if primary.valid_area_ha <= 0:
        return not_applicable(
            "4.1 Pathway Distribution",
            "The pathway layer does not cover this project area, so no pathway can be "
            "recommended.",
        )

    # Denominator is the whole site. Code 0 and nodata are absorbed by an Unclassified row, so
    # the table sums to 100 of the AOI rather than of the covered part.
    rows: list[PathwayShare] = []
    classified_ha = 0.0
    for code, label in PATHWAY_CODES.items():
        if code == 0:
            continue  # 0 is folded into Unclassified below, together with nodata
        area_ha = int((primary.values == code).sum()) * primary.pixel_area_ha
        classified_ha += area_ha
        rows.append(
            PathwayShare(
                code=code,
                label=label,
                area_ha=area_ha,
                pct=safe_pct(area_ha, aoi.area_ha),
                is_pathway=code in PATHWAY_ELIGIBLE_CODES,
            )
        )

    unclassified_ha = max(0.0, aoi.area_ha - classified_ha)
    rows.append(
        PathwayShare(
            code=0,
            label=UNCLASSIFIED_LABEL,
            area_ha=unclassified_ha,
            pct=safe_pct(unclassified_ha, aoi.area_ha),
            is_pathway=False,
        )
    )

    eligible_ha = sum(r.area_ha for r in rows if r.is_pathway)
    eligible_pct = safe_pct(eligible_ha, aoi.area_ha)

    flags: list[str] = []
    unclassified_pct = safe_pct(unclassified_ha, aoi.area_ha)
    if unclassified_pct > PATHWAY_UNCLASSIFIED_WARN_PCT:
        flags.append(
            f"4.1: {unclassified_pct:.0f}% of the AOI carries no pathway value. Every share "
            "below describes only the remainder of the site."
        )

    # Bands 2 and 3 pass through untabulated, for the activity generator in F02-P5.
    ecosystem = _codes_present(aoi, PATHWAY_ECOSYSTEM_BAND, PATHWAY_ECOSYSTEM_CODES)
    catcode = _codes_present(aoi, PATHWAY_CATCODE_BAND, PATHWAY_CATCODE_LABELS)

    pathway_rows = sort_by_area([r for r in rows if r.is_pathway and r.area_ha > 0])
    return ComponentResult(
        component="4.1 Pathway Distribution",
        applicable=True,
        narrative="",  # no rendered sentence: downstream reads values, the human reads the table
        tables={"pathway_distribution": rows},  # every code plus Unclassified, sums to 100
        values={
            "chart_series": "pathway_distribution",
            "chart_unit": "%",
            "chart_axis_label": "Share of project area (%)",
            "eligible_ha": eligible_ha,
            "eligible_pct": eligible_pct,
            "pathway_ha": {r.label: r.area_ha for r in rows if r.is_pathway},
            "dominant_pathway": pathway_rows[0].label if pathway_rows else None,
            "unclassified_pct": unclassified_pct,
            # Band 2 ecosystem and band 3 cat_code, for the F02-P5 activity join on
            # (cat_code, ecosystem). Four class ecosystem, NOT the three class layer of 1.1.
            "reference_ecosystem_codes": ecosystem["codes"],
            "reference_ecosystem_labels": ecosystem["labels"],
            "cat_code_codes": catcode["codes"],
            "cat_code_labels": catcode["labels"],
        },
        flags=flags,
    )


results["4.1"] = analyze_pathway_distribution(aoi)
show_result(results["4.1"])

---
## 4.2 Activity List

The activities that apply to the AOI, derived from 4.1. For each category present in the AOI it
lists the activities from the canonical_v3 catalog.

**How.** Bands 3 (`cat_code`) and 2 (`ecosystem`) are read on the same grid and cross-tabulated
into the unique `(cat_code, ecosystem)` pairs present, with area. Each pair is joined to
`canonical_v3_activities` (config `ACTIVITY_TABLE`, keyed on `(cat_code, ecosystem)`). Ineligible
categories carry no activity by design; a category with no matching catalog row is flagged, not
dropped.

**Data.** `ACTIVITY_TABLE`, exported from the "NBS Pathway Logic" Sheet tab
canonical_v3_activities. Set its path in `config.py`; the expected columns are documented there.
The full activity rows, including the Triple Win benefits and the two carbon QB flags, travel in
`values["by_category"]` for F02-P5; the displayed table shows the activity list only.

In [9]:
def analyze_activity_list(aoi: AOI) -> ComponentResult:
    """Component 4.2. Activities per (cat_code, ecosystem) category present in the AOI."""
    # Bands 3 and 2 on one shared grid, so each pixel pairs its own cat_code with its ecosystem.
    catcode = load_raster_clipped(PATHWAY_RASTER, aoi, resampling="nearest",
                                  band=PATHWAY_CATCODE_BAND)
    ecosystem = load_raster_clipped(PATHWAY_RASTER, aoi, resampling="nearest",
                                    band=PATHWAY_ECOSYSTEM_BAND, like=catcode)

    cat, eco = catcode.values, ecosystem.values
    valid = ~np.ma.getmaskarray(cat) & ~np.ma.getmaskarray(eco)
    if not valid.any():
        return not_applicable(
            "4.2 Activity List",
            "The pathway layer does not cover this project area, so no activities apply.",
        )

    # Unique (cat_code, ecosystem) pairs present, with pixel count -> area.
    pairs = np.stack([cat[valid].astype(int), eco[valid].astype(int)], axis=1)
    uniq, counts = np.unique(pairs, axis=0, return_counts=True)
    px_area = catcode.pixel_area_ha

    table = load_activity_table(ACTIVITY_TABLE)

    # Dominant categories first.
    order = np.argsort(counts)[::-1]
    rows: list[dict] = []
    by_category: dict[str, dict] = {}
    flags: list[str] = []

    for idx in order:
        cc, ec = int(uniq[idx][0]), int(uniq[idx][1])
        if cc == 0:
            continue  # mask
        area_ha = int(counts[idx]) * px_area
        pathway_code = PATHWAY_CATCODE_TO_PATHWAY.get(cc)
        pathway = PATHWAY_CODES.get(pathway_code, "Unknown")
        cat_label = PATHWAY_CATCODE_LABELS.get(cc, f"cat {cc}")
        eco_label = PATHWAY_ECOSYSTEM_CODES.get(ec, f"ecosystem {ec}")

        if ec == 0:
            acts = []
            flags.append(
                f"4.2: {cat_label} appears with ecosystem 0 (no reference) on "
                f"{fmt_ha(area_ha)}; the pathway script should have masked these pixels."
            )
        elif pathway_code == 4:
            acts = []  # Ineligible: no activity by design
        else:
            acts = table.get((cc, ec), [])
            if not acts:
                flags.append(
                    f"4.2: no catalog row for ({cat_label}, {eco_label}); "
                    f"{fmt_ha(area_ha)} left without an activity."
                )

        if acts:
            for a in acts:
                rows.append({
                    "pathway": pathway, "category": cat_label, "ecosystem": eco_label,
                    "area_ha": round(area_ha, 1),
                    "activity_id": a["activity_id"], "activity": a["activity"],
                })
        else:
            note = "(ineligible, no activity)" if pathway_code == 4 else "(no catalog match)"
            rows.append({
                "pathway": pathway, "category": cat_label, "ecosystem": eco_label,
                "area_ha": round(area_ha, 1), "activity_id": "", "activity": note,
            })

        by_category[f"{cat_label} | {eco_label}"] = {
            "pathway": pathway, "area_ha": area_ha, "activities": acts,
        }

    return ComponentResult(
        component="4.2 Activity List",
        applicable=True,
        narrative="",
        tables={"activities": rows},   # one row per activity, category by category
        values={
            "by_category": by_category,        # full rows incl. benefits + QB flags, for F02-P5
            "category_count": len(by_category),
            "activity_count": sum(1 for r in rows if r["activity_id"]),
        },
        flags=flags,
    )


results["4.2"] = analyze_activity_list(aoi)
show_result(results["4.2"])

---
## Save

Each section above already ran and displayed itself. This cell writes them to one combined JSON.

In [11]:
path = save_results(results, aoi, aoi_id, STAGE_PATHWAY)
print(f"Saved {path}")

Saved outputs\aoi1__F02-P4-pathway.json
